# URA EXACT — Remote LLM server (Colab GPU + Ollama + Cloudflare Tunnel)

Serves an **OpenAI-compatible** LLM on a Colab/Kaggle **T4** GPU using **Ollama**
(llama.cpp CUDA) and exposes it through a **free Cloudflare Quick Tunnel** (no
account, no token). Copy the printed `https://*.trycloudflare.com/v1` URL into
your local `.env` as `URA_LLM_BASE_URL`.

**Why Ollama and not vLLM here?** On a T4 (compute capability 7.5) vLLM 0.22.x
forces the **FLASHINFER** attention backend (it ignores
`VLLM_ATTENTION_BACKEND`), whose paged kernel crashes at decode
(`BatchPrefillWithPagedKVCache failed ... invalid argument`); and uninstalling
flashinfer makes the V1 engine fail to initialize. Ollama uses llama.cpp CUDA
kernels that run fine on sm 7.5, sidestepping the whole problem. The output is
still an OpenAI-compatible `/v1` endpoint, so the local backend only needs the URL.

**Runtime → Change runtime type → GPU (T4)** before running. Run cells top to bottom.

## 1. Configuration

`MODEL` is an **Ollama tag** (not a HuggingFace repo). `qwen2.5:7b-instruct` pulls
the Q4_K_M GGUF (~4.7GB), which fits a 16GB T4 with room to spare. Smaller/larger
options are listed inline.

In [ ]:
# ============================ CONFIG ============================
# Ollama model tags (https://ollama.com/library/qwen2.5). Default quant is Q4_K_M.
#   qwen2.5:7b-instruct        ~4.7GB  (default; strong translator, fits T4)
#   qwen2.5:7b-instruct-q5_K_M ~5.4GB  (a bit higher quality)
#   qwen2.5:3b-instruct        ~1.9GB  (fast fallback)
#   qwen2.5:14b-instruct       ~9GB    (only if you have a bigger GPU; NOT a T4)
MODEL = "qwen2.5:7b-instruct"

PORT = 11434            # Ollama default port
CTX = 4096              # context window (num_ctx)

print('Model :', MODEL)
print('Port  :', PORT)
print('Ctx   :', CTX)

## 2. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader || echo 'NO GPU — set Runtime > Change runtime type > GPU'

## 3. Install Ollama + cloudflared

Ollama is a single static install (CUDA build). cloudflared is a single static
binary (free Quick Tunnels, no login). No Python/transformers/numpy ABI issues.

In [ ]:
# Ollama (official install script; installs the CUDA-enabled server binary).
# The installer extracts a zstd-compressed tarball, so zstd must be present
# first (Colab base image lacks it — without it you get
# 'ERROR: This version requires zstd for extraction' and `ollama: command not found`).
# pciutils (lspci) + lshw let the installer DETECT the T4 GPU; without them you
# get 'WARNING: Unable to detect NVIDIA/AMD GPU' (Ollama may skip GPU setup).
# The 'W: Skipping acquire ... r2u.stat.illinois.edu' line and 'systemd is not
# running' warning are harmless Colab noise — not errors.
!apt-get -qq update 2>/dev/null; apt-get -qq install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh
# Confirm the binary is installed. Do NOT run `ollama --version` here: with no
# server running it tries to reach one, hangs, and needs Ctrl-C (the `^C` you
# saw). The server is started in cell 4; `which ollama` is enough to verify.
!which ollama && echo 'OLLAMA INSTALLED OK' || echo 'INSTALL FAILED — see log above'

# cloudflared static binary (free Quick Tunnels — no login).
# Stop any running instance first, else overwriting the binary fails with
# 'Text file busy'.
!pkill -f cloudflared 2>/dev/null; sleep 1
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

## 4. Start the Ollama server (background) and pull the model

`OLLAMA_HOST=0.0.0.0:PORT` makes it bind all interfaces so the tunnel can reach
it. The first run downloads the model GGUF. Wait for `Ollama is READY` and the
pull to finish before running the tunnel cell.

In [ ]:
import os, subprocess, time, itertools, urllib.request, json

env = os.environ.copy()
env['OLLAMA_HOST'] = f'0.0.0.0:{PORT}'
# Keep the model resident so it doesn't unload between requests during a benchmark.
env['OLLAMA_KEEP_ALIVE'] = '-1'

srv_log = open('ollama.log', 'w')
srv_proc = subprocess.Popen(['ollama', 'serve'], stdout=srv_log, stderr=subprocess.STDOUT, env=env)
print('ollama serve PID:', srv_proc.pid)

base = f'http://127.0.0.1:{PORT}'
ready = False
for i in range(60):
    if srv_proc.poll() is not None:
        print('ollama serve exited early. Log tail:')
        print(open('ollama.log').read()[-2000:])
        break
    try:
        with urllib.request.urlopen(base + '/api/version', timeout=3) as r:
            if r.status == 200:
                print('Ollama is READY after %ds:' % (i * 2), json.loads(r.read()))
                ready = True
                break
    except Exception:
        pass
    time.sleep(2)

# Pull the model (streams progress to stdout).
if ready:
    print('\nPulling', MODEL, '(first time downloads the GGUF)...')
    subprocess.run(['ollama', 'pull', MODEL], env=env, check=False)
    # Confirm it is registered.
    out = subprocess.run(['ollama', 'list'], env=env, capture_output=True, text=True)
    print(out.stdout)

## 4b. Local generation self-test (before opening the tunnel)

Hits Ollama's OpenAI-compatible endpoint on localhost to confirm **generation**
works. First call also loads the model into VRAM (a few seconds).

In [ ]:
import urllib.request, json, time as _t
_url = f'http://127.0.0.1:{PORT}/v1/chat/completions'
_payload = {
    'model': MODEL,
    'messages': [
        {'role': 'system', 'content': 'Reply with JSON only.'},
        {'role': 'user', 'content': 'Return {"answer":"yes"} if 2+2=4.'},
    ],
    'max_tokens': 40, 'temperature': 0.0,
}
try:
    _req = urllib.request.Request(_url, data=json.dumps(_payload).encode(),
                                  headers={'Content-Type': 'application/json'})
    _s = _t.time()
    with urllib.request.urlopen(_req, timeout=180) as r:
        _out = json.loads(r.read())
    print('LOCAL GEN OK (%.1fs):' % (_t.time() - _s), _out['choices'][0]['message']['content'])
except Exception as e:
    _body = e.read().decode()[:600] if hasattr(e, 'read') else ''
    print('LOCAL GEN FAIL:', type(e).__name__, e)
    print('body:', _body)
    print('--- ollama.log tail ---')
    print(open('ollama.log').read()[-2000:])

## 5. Open the free Cloudflare Quick Tunnel and print the public URL

In [ ]:
import subprocess, re, time

tunnel_log = open('cloudflared.log', 'w')
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--no-autoupdate', '--url', f'http://localhost:{PORT}'],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)
print('cloudflared PID:', tunnel_proc.pid)

public_url = None
for i in range(60):
    time.sleep(2)
    try:
        log = open('cloudflared.log').read()
    except Exception:
        log = ''
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', log)
    if m:
        public_url = m.group(0)
        break
    if tunnel_proc.poll() is not None:
        print('cloudflared exited. Log:'); print(log[-2000:]); break

if public_url:
    llm_base = public_url + '/v1'
    print('\n' + '=' * 70)
    print('PUBLIC LLM ENDPOINT (Cloudflare Quick Tunnel):')
    print('  ' + llm_base)
    print('=' * 70)
    print('\nPaste these into your LOCAL .env (or export before starting the API):\n')
    print(f'URA_LLM_BASE_URL={llm_base}')
    print(f'URA_LLM_MODEL={MODEL}')
    print('\nThen restart the local API server so it picks up the new endpoint.')
else:
    print('Could not obtain a tunnel URL. See cloudflared.log above.')

## 6. Smoke test through the public tunnel

Confirms the full path (tunnel → Ollama) works and returns JSON.

In [ ]:
import urllib.request, json, time as _t
assert public_url, 'No tunnel URL — run cell 5 first.'
url = public_url + '/v1/chat/completions'
payload = {
    'model': MODEL,
    'messages': [
        {'role': 'system', 'content': 'Reply with JSON only.'},
        {'role': 'user', 'content': 'Return {"answer":"yes"} if 2+2=4.'},
    ],
    'max_tokens': 40, 'temperature': 0.0,
}
req = urllib.request.Request(url, data=json.dumps(payload).encode(),
                             headers={'Content-Type': 'application/json'})
_s = _t.time()
with urllib.request.urlopen(req, timeout=180) as r:
    out = json.loads(r.read())
print('latency: %.1fs' % (_t.time() - _s))
print('content:', out['choices'][0]['message']['content'])

## 7. Keep alive

Leave this notebook tab open while you run local benchmarks. If the tunnel URL
changes (Colab restart/disconnect), re-run cells 4–5 and update `URA_LLM_BASE_URL`
locally with the new URL.

In [ ]:
import time, datetime
print('Heartbeat — keep this running. Endpoint:', (public_url + '/v1') if public_url else 'N/A')
try:
    while True:
        time.sleep(120)
        alive_s = (srv_proc.poll() is None)
        alive_t = (tunnel_proc.poll() is None)
        print(datetime.datetime.now().strftime('%H:%M:%S'), 'ollama:', 'up' if alive_s else 'DOWN', '| tunnel:', 'up' if alive_t else 'DOWN')
        if not alive_s or not alive_t:
            print('A process died — re-run cells 4/5.'); break
except KeyboardInterrupt:
    print('Stopped heartbeat.')

## (Optional) Restart only the Ollama server

Use this if `srv_proc` died (Colab idle/OOM) but the tunnel is still alive.
Re-runs cell 4 logic in place WITHOUT touching the Cloudflare tunnel, so the
public URL stays the same and you don't need to update `URA_LLM_BASE_URL`.

In [ ]:
import os, subprocess, time, urllib.request, json
if 'srv_proc' in globals() and srv_proc.poll() is None:
    print('Ollama already up (PID', srv_proc.pid, ') — nothing to do.')
else:
    env = os.environ.copy()
    env['OLLAMA_HOST'] = f'0.0.0.0:{PORT}'
    env['OLLAMA_KEEP_ALIVE'] = '-1'
    srv_log = open('ollama.log', 'a')   # append, preserve old logs
    srv_proc = subprocess.Popen(['ollama', 'serve'], stdout=srv_log, stderr=subprocess.STDOUT, env=env)
    print('Re-spawned Ollama PID:', srv_proc.pid)
    base = f'http://127.0.0.1:{PORT}'
    for i in range(60):
        try:
            with urllib.request.urlopen(base + '/api/version', timeout=3) as r:
                if r.status == 200:
                    print('Ollama is READY again after %ds:' % (i*2), json.loads(r.read())); break
        except Exception:
            time.sleep(2)
    else:
        print('Ollama did not come up in 120s — check ollama.log tail:'); print(open('ollama.log').read()[-2000:])

## (Danger) Stop everything

**Do NOT run this while you still need the tunnel.** Stopping `srv_proc` kills
Ollama and any in-flight `/v1/chat/completions` request gets a 530 / Connection
refused at the client. To prevent accidental clicks, this cell requires you to
explicitly set `CONFIRM_STOP = True` below before running.

In [ ]:
CONFIRM_STOP = False   # flip to True ONLY when you really want to tear down

if not CONFIRM_STOP:
    print('Refusing to stop — set CONFIRM_STOP = True above and re-run.')
else:
    for p in ['srv_proc', 'tunnel_proc']:
        try:
            globals()[p].terminate(); print('terminated', p)
        except Exception as e:
            print('skip', p, e)